<a href="https://colab.research.google.com/github/babi00/ai4biological-pattern/blob/guido-clean/invasive_plants_analysis/predictions_with_logits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We use each model to predict the output (invasive, non-invasive) for every species using the model that has been trained on everything except that species.

We only work on the subset of regions that have the pair of traits from the RF, and we save the logits of the prediction to compare the confidence between the two datasets (images complete and images with holes)

In [2]:
#@title Imports and downloads
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import functional as TF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision import transforms, models
from sklearn.metrics import classification_report, multilabel_confusion_matrix, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import GroupKFold
from google.colab import drive
import time
from tqdm import tqdm
!pip install open_clip_torch
import open_clip
import math
from collections import Counter
import datetime
import json
import os
import random

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

drive.mount('/content/drive')

# Clone the repository and checkout the 'clean-barbara' branch
!git clone --branch clean-barbara https://github.com/babi00/ai4biological-pattern.git
%cd ai4biological-pattern

# Enable sparse checkout
!git sparse-checkout init --cone

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'ai4biological-pattern'...
remote: Enumerating objects: 735244, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 735244 (delta 92), reused 143 (delta 71), pack-reused 735069 (from 3)
Receiving objects: 100% (735244/735244), 8.79 GiB | 51.45 MiB/s, done.
Resolving deltas: 100% (2296/2296), done.
Updating files: 100% (214837/214837), done.
/content/ai4biological-pattern
Updating files: 100% (214835/214835), done.


In [3]:
#@title Training log saving function
def save_training_log(model_name, output_dir, train_losses, val_losses, train_accuracies, val_accuracies, classification_report_dict):
    import datetime
    import json
    import os

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_dir = f"/content/drive/MyDrive/Thesis/{output_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "training_log.json")

    # Load existing logs (if any)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    # Prepare this run's data
    this_log = {
        "model_name": model_name,
        "timestamp": timestamp,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "final_val_loss": val_losses[-1] if val_losses else None,
        "final_train_accuracy": train_accuracies[-1] if train_accuracies else None,
        "final_val_accuracy": val_accuracies[-1] if val_accuracies else None,
        "classification_report_last_epoch": classification_report_dict[-1] if isinstance(classification_report_dict, list) else classification_report_dict,
        "train_loss_history": train_losses,
        "val_loss_history": val_losses,
        "train_accuracy_history": train_accuracies,
        "val_accuracy_history": val_accuracies
    }

    all_logs.append(this_log)

    # Save updated list
    with open(log_path, "w") as f:
        json.dump(all_logs, f, indent=4)

    print(f"📝 Appended log to {log_path}")


In [4]:
#@title Horizontal flip
def hflip(image):
  return TF.hflip(image)

In [5]:
#@title Original Dataset
class InvasiveSpeciesDataset(Dataset):
    def __init__(self, root_dir, transform=None, get_label_fn=None):
        self.root_dir = root_dir
        self.transform = transform
        self.entries = []
        self.label_map = {
            "Invasive" : 0,
            "Non Invasive" : 1
        }

        taxa_folders = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) and d != "metadata"]
        for taxon in taxa_folders:
            meta_path = os.path.join(root_dir, "filtered_metadata", f"{taxon}_metadata.csv")
            if not os.path.exists(meta_path):
                continue
            metadata_df = pd.read_csv(meta_path)
            for idx, row in metadata_df.iterrows():
              if pd.isna(row["filename"]):
                  print(f"⚠️ Missing filename in taxon '{taxon}' at index {idx}")
                  continue
              self.entries.append((taxon, row))

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        taxon, row = self.entries[idx]
        filename = str(row["filename"])
        image_path = os.path.join(self.root_dir, taxon, filename)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.label_map.get(row["label"])
        return image, label, idx, taxon

In [6]:
#@title Augmented Dataset
class AugmentedDataset(Dataset):
  def __init__(self, original_dataset, transform=None):
    self.original_dataset = original_dataset
    self.transform = transform
    self.entries = [entry for entry in original_dataset.entries]
    self.root_dir = original_dataset.root_dir
    self.label_map = {
            "Invasive" : 0,
            "Non Invasive" : 1
        }

  def __len__(self):
    return len(self.entries)

  def __getitem__(self, idx):
    taxon, row = self.entries[idx]
    filename = str(row['filename'])
    image_path = os.path.join(self.root_dir, taxon, filename)
    image = Image.open(image_path).convert("RGB")
    if self.transform:
        image = self.transform(image)
    label = self.label_map.get(row["label"])
    return image, label, idx, taxon

In [7]:
#@title Augmented Dataset (Only for lythrum virgatum)
class AugmentedVirgatumDataset(Dataset):
  def __init__(self, original_dataset, transform=None):
    self.original_dataset = original_dataset
    self.transform = transform
    self.entries = [entry for entry in original_dataset.entries if entry[0] == 'lythrum_virgatum']
    self.root_dir = original_dataset.root_dir
    self.label_map = {
            "Invasive" : 0,
            "Non Invasive" : 1
        }

  def __len__(self):
    return len(self.entries)

  def __getitem__(self, idx):
    taxon, row = self.entries[idx]
    filename = str(row['filename'])
    image_path = os.path.join(self.root_dir, taxon, filename)
    image = Image.open(image_path).convert("RGB")
    if self.transform:
        image = self.transform(image)
    label = self.label_map.get(row["label"])
    return image, label, idx, taxon

In [8]:
#@title Return the dataloader of all images and not already separated (also class weights are not present)
def get_data_loader(root_dir, batch_size=32, use_bioclip=True, preprocess=None):
    if use_bioclip and preprocess:
        transform = preprocess
    else:
        transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor()
        ])

    dataset = InvasiveSpeciesDataset(
        root_dir=root_dir,
        transform=transform,
    )

    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    return loader

In [9]:
#@title Augmentation helper functions
def get_augmented_transform_flipping(preprocess=None):

    if preprocess:
        # BioCLIP: use its own preprocessing, plus flipping
        return transforms.Compose([
            transforms.Lambda(lambda img: TF.hflip(img)),
            preprocess
        ])
    else:
        # ResNet or default: manual transform
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Lambda(lambda img: TF.hflip(img)),
            transforms.ToTensor()
        ])


def get_augmented_transform_jitter(preprocess=None):

    if preprocess:
        # BioCLIP: use its own preprocessing, plus jittering (no hue modification)
        return transforms.Compose([
            transforms.ColorJitter(0.3, 0.3, 0.3),
            preprocess
        ])
    else:
        # ResNet or default: manual transform
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ColorJitter(0.3, 0.3, 0.3),
            transforms.ToTensor()
        ])

In [10]:
#@title Augmented DataLoader (N.B. Modify the AugmentedDataset with what you need)
def get_data_loader_augmented(root_dir, batch_size=32, use_bioclip=True, preprocess=None, augmentation='flipping'):
    if use_bioclip and preprocess:
        original_transform = preprocess
    else:
        original_transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor(),
          # transforms.RandomHorizontalFlip(p=1.0)
        ])

    if augmentation=='flipping':
      augmented_transform = get_augmented_transform_flipping(preprocess if use_bioclip else None)
    elif augmentation=='jittering':
      augmented_transform = get_augmented_transform_jitter(preprocess if use_bioclip else None)
    else:
      print("Error in the augmentation!")
      augmented_transform = -1 #this will break after


    original_dataset = InvasiveSpeciesDataset(
        root_dir=root_dir,
        transform=original_transform,
    )

    augmented_dataset = AugmentedVirgatumDataset(
        original_dataset=original_dataset,
        transform=augmented_transform,
    )

    #this is confirmed to print the correct entries
    # print("### INSIDE GET DATA LOADER - AUGMENTED DATASET ENTRIES: ", augmented_dataset.entries)

    augmented_loader = DataLoader(augmented_dataset, batch_size=32, shuffle=False)

    return augmented_loader

In [11]:
#@title 2.Define the embedding extractor:
#ResNet without the last FC layer
def get_resnet_embeddings_extractor_model():
    #Use a pre-trained ResNet18 model
    model = models.resnet18(weights='IMAGENET1K_V1')

    #Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    embeddings_extractor= nn.Sequential(*list(model.children())[:-1]) #removes the last FC layer
    print("Model obtained...")

    return embeddings_extractor

#BioCLIP model
def get_bioclip_embeddings_extractor_model(device, bioclip_version=2):
    if bioclip_version==2:
        print("BioCLIP 2!")
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
    else:
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
    print("Model obtained...")
    model.to(device)
    model.eval()
    return model, preprocess

In [12]:
#@title 3. Extract embeddings using the embeddings_extractor and return the embeddings tensors
def extract_embeddings(dataloader, embeddings_extractor, device, embeddings_save_path, augmented=False, use_bioclip=True, dataset=None):
    embeddings_extractor.eval()
    all_embeddings = []
    all_labels = []
    all_ids = []
    all_groups = []  # ✅ NEW: store taxa names

    with torch.no_grad():
      for images, labels, obs_id, taxon in tqdm(dataloader, desc='Extracting embeddings'):
          images = images.to(device)
          if use_bioclip:
              embeddings = embeddings_extractor.encode_image(images)
              embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)  # Optional: normalize
          else:
              embeddings = embeddings_extractor(images).view(images.size(0), -1)

          all_embeddings.append(embeddings.cpu())
          all_labels.append(labels)
          all_ids.extend(obs_id)

          all_groups.extend(taxon) #remember to use extend and not append to get a flat list and not a list of tuples of the batch

          # ✅ OLD: reconstruct taxa names directly here
          # if dataset is not None:
          #     for i in obs_id:
          #         taxon, _ = dataset.entries[int(i)]
          #         all_groups.append(taxon)

    embeddings_tensors = torch.cat(all_embeddings, dim=0).to(device)
    labels_tensors = torch.cat(all_labels, dim=0).to(device)
    obs_id_tensor = torch.tensor(all_ids).to(device)

    all_embeddings = {
       'embeddings' : embeddings_tensors,
       'labels' : labels_tensors,
       'ids' : obs_id_tensor,
       'groups': all_groups  # ✅ Save group names
    }

    if augmented==False:
      torch.save(all_embeddings, embeddings_save_path)

    # embedding_size = embeddings_tensors.shape[1] #useful for classifier

    return all_embeddings

In [13]:
#@title Split the embeddings 80-20, then add the augmentation to the training set
def split_embeddings(embeddings_dict, augmented_embeddings_dict=None, val_percentage=0.2):

    """Split the embeddings and not the images into training set and validation set"""

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    embedding_size = embeddings_tensors.shape[1] #useful for classifier

    dataset = torch.utils.data.TensorDataset(embeddings_tensors, labels_tensors, obs_id_tensor)

    train_size = int((1-val_percentage) * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    #if using augmentation
    if augmented_embeddings_dict:
      print("Augmenting train dataset...")
      aug_embeddings_tensors = augmented_embeddings_dict["embeddings"]
      aug_labels_tensors = augmented_embeddings_dict["labels"]
      aug_obs_id_tensor = augmented_embeddings_dict["ids"]

      augmented_train_dataset = torch.utils.data.TensorDataset(aug_embeddings_tensors, aug_labels_tensors, aug_obs_id_tensor)

      #concatenate train dataset with augmented train dataset
      train_dataset = ConcatDataset([train_dataset, augmented_train_dataset])

    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)

    # total = sum(label_counts.values())
    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts[0]),
        1.0 / math.log(1.02 + label_counts[1]),
    ], dtype=torch.float)

    print("⚖️ Class weights:", class_weights)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True)

    return train_loader, val_loader, class_weights, embedding_size



In [14]:
#@title K fold helper function

def inspect_folds_groups(dataset, n_splits=5):
    """
    Print the number of validation groups, their names,
    and the class distribution for each fold (GroupKFold).
    """
    labels = [dataset.label_map[row["label"]] for _, row in dataset.entries]
    groups = [taxon for taxon, _ in dataset.entries]

    gkf = GroupKFold(n_splits=n_splits)

    print(f"🔍 Inspecting {n_splits}-fold GroupKFold:")
    for fold_idx, (_, val_idx) in enumerate(gkf.split(np.zeros(len(labels)), labels, groups)):
        val_groups = sorted(set(groups[i] for i in val_idx))
        val_labels = [labels[i] for i in val_idx]
        class_counts = Counter(val_labels)

        print(f"\n📂 Fold {fold_idx}: {len(val_groups)} validation groups")
        print(f"Class distribution: {dict(class_counts)}")
        print(f"Species in this fold: {val_groups}")



def inspect_folds_groups_with_augmentation(embeddings_dict, augmented_embeddings_dict=None, dataset=None, n_splits=5, remove_species=None):
    labels = embeddings_dict["labels"].cpu().numpy()
    groups = np.array(embeddings_dict["groups"])
    # print("### Inside  INSPECT_FOLDS_GROUPS_WITH AUGMENTATION - embeddings_dict['groups'])", embeddings_dict['groups'][:5]) correct a list of strings
    # print(len(groups)) 45361 correct


    if remove_species is not None:
        indices = np.array([g not in remove_species for g in groups])
        labels = labels[indices]
        groups = groups[indices]

    gkf = GroupKFold(n_splits=n_splits)

    print(f"🔍 Inspecting {n_splits}-fold GroupKFold with augmentation effects:")
    print(f"📊 Total samples in filtered embeddings: {len(labels)}")
    print(f"📊 Total unique groups: {len(set(groups))}")

    if augmented_embeddings_dict:
        # print("### Inside  INSPECT_FOLDS_GROUPS_WITH AUGMENTATION - augmented_embeddings_dict['groups']", augmented_embeddings_dict["groups"])
        aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
        aug_groups = np.array(augmented_embeddings_dict["groups"])
        if remove_species is not None:
            aug_indices = np.array([g not in remove_species for g in aug_groups])
            aug_labels = aug_labels[aug_indices]

        print(f"📊 Total augmented samples (filtered): {len(aug_labels)}")
        print(f"📊 Augmented class distribution: {dict(Counter(aug_labels))}")

    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(np.zeros(len(labels)), labels, groups)):
        val_groups = sorted(set(groups[val_idx]))
        val_labels = labels[val_idx]
        train_labels = labels[train_idx]

        print(f"\n📂 Fold {fold_idx}: {len(val_groups)} validation groups")
        print(f"VALIDATION - Class distribution: {dict(Counter(val_labels))}")
        print(f"TRAINING (original) - Class distribution: {dict(Counter(train_labels))}")

        if augmented_embeddings_dict:
            aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
            aug_groups = np.array(augmented_embeddings_dict["groups"])
            if remove_species is not None:
                aug_labels = aug_labels[[g not in remove_species for g in aug_groups]]
            combined_train_labels = list(train_labels) + list(aug_labels)
            print(f"TRAINING (with augmentation) - Class distribution: {dict(Counter(combined_train_labels))}")
        print(f"Species in validation fold: {val_groups}")


#E.g. Validating fold 0 (salicaria) without hyssopifolia
#without augmentation it trains on 709 embeddings
#Augmenting virgatum is 68 embeddings per augmentation
#So if i augment x3 I expect it to train on 845 total embeddings
def debug_validation_set_actual_with_augmentation(embeddings_dict, augmented_embeddings_dict=None, dataset=None, n_splits=5, fold_idx=0, remove_species=None):
    labels = embeddings_dict["labels"].cpu().numpy()
    groups = np.array(embeddings_dict["groups"])

    if remove_species is not None:
        keep_mask = np.array([g not in remove_species for g in groups])
        labels = labels[keep_mask]
        groups = groups[keep_mask]

    #this will go into the model name. Generalize to account for the possibility of having splits with more than one species
    taxa = ''

    for i, t in enumerate(sorted(set(groups))):
      taxa = taxa + str(t)

      if i!=len(groups)-1:
        taxa = taxa  + '_'

    gkf = GroupKFold(n_splits=n_splits)
    all_splits = list(gkf.split(np.zeros(len(labels)), labels, groups))
    train_idx, val_idx = all_splits[fold_idx]

    val_labels = labels[val_idx]
    val_groups = groups[val_idx]
    train_labels = labels[train_idx]
    train_groups = groups[train_idx]

    #this will go into the model name.

    taxa = str(sorted(set(val_groups))[0])

    print(f"\n🔍 DEBUG: Actual sets for fold {fold_idx} ({taxa}):")
    print(f"VALIDATION SET:")
    print(f"  Size: {len(val_labels)}")
    print(f"  Class distribution: {dict(Counter(val_labels))}")
    print(f"  Unique groups: {sorted(set(val_groups))}")

    print(f"TRAINING SET (original):")
    print(f"  Size: {len(train_labels)}")
    print(f"  Class distribution: {dict(Counter(train_labels))}")
    print(f"  Unique groups: {sorted(set(train_groups))}")

    if augmented_embeddings_dict:
        aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
        aug_groups = np.array(augmented_embeddings_dict["groups"])

        # Exclude both remove_species and validation species
        val_groups_set = set(val_groups)
        aug_mask = np.array([
            (g not in remove_species) and (g not in val_groups_set)
            for g in aug_groups
        ])

        aug_labels = aug_labels[aug_mask]

        combined_train_labels = list(train_labels) + list(aug_labels)
        print(f"TRAINING SET (with augmentation):")
        print(f"  Size: {len(combined_train_labels)}")
        print(f"  Class distribution: {dict(Counter(combined_train_labels))}")

    return taxa

In [15]:
#@title Group K Fold

from sklearn.model_selection import GroupKFold
from torch import from_numpy

def split_embeddings_groupkfold(embeddings_dict, augmented_embeddings_dict=None, n_splits=5, fold_idx=0, remove_species=None):
    """
    Split the embeddings using GroupKFold instead of random_split.
    Automatically uses taxon as groups (reconstructed from dataset.entries).
    """

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    ###
    #modified to exclude species from the training

    embeddings_np = embeddings_tensors.cpu().numpy()
    labels_np = labels_tensors.cpu().numpy()
    obs_id_np = obs_id_tensor.cpu().numpy()
    groups_np = np.array(embeddings_dict["groups"])  # ✅ Directly use stored groups

    if remove_species != None:
      #mask
      indices = [groups_np[i] not in remove_species for i in range(len(groups_np))]

      embeddings_np_reduced = np.array([embeddings_np[i] for i in range(len(indices)) if indices[i]])
      labels_np_reduced = np.array([labels_np[i] for i in range(len(indices)) if indices[i]])
      obs_id_np_reduced = np.array([obs_id_np[i] for i in range(len(indices)) if indices[i]])
      groups_np_reduced = [groups_np[i] for i in range(len(indices)) if indices[i]]

    else:
      embeddings_np_reduced = embeddings_np
      labels_np_reduced = labels_np
      obs_id_np_reduced = obs_id_np
      groups_np_reduced = groups_np

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # or get from caller
    embeddings_tensors = torch.from_numpy(np.stack(embeddings_np_reduced)).to(device)
    labels_tensors = torch.from_numpy(np.array(labels_np_reduced)).to(device)
    obs_id_tensor = torch.from_numpy(np.array(obs_id_np_reduced)).to(device)

    #end of modification
    ###

    embedding_size = embeddings_tensors.shape[1]  # useful for classifier

    labels_np = labels_tensors.cpu().numpy()

    gkf = GroupKFold(n_splits=n_splits)
    all_splits = list(gkf.split(np.zeros(len(labels_np_reduced)), labels_np_reduced, groups_np_reduced))
    train_idx, val_idx = all_splits[fold_idx]

    #build datasets
    train_dataset = torch.utils.data.TensorDataset(
        embeddings_tensors[train_idx],
        labels_tensors[train_idx],
        obs_id_tensor[train_idx]
    )
    val_dataset = torch.utils.data.TensorDataset(
        embeddings_tensors[val_idx],
        labels_tensors[val_idx],
        obs_id_tensor[val_idx]
    )

    #augmentation
    if augmented_embeddings_dict:
        # print("elements in augmented dict: ", len(augmented_embeddings_dict['embeddings']))
        print("Augmenting train dataset...")
        aug_embeddings_tensors = augmented_embeddings_dict["embeddings"]
        aug_labels_tensors = augmented_embeddings_dict["labels"]
        aug_obs_id_tensor = augmented_embeddings_dict["ids"]

        aug_embeddings_np = aug_embeddings_tensors.cpu().numpy()
        aug_labels_np = aug_labels_tensors.cpu().numpy()
        aug_obs_id_np = aug_obs_id_tensor.cpu().numpy()
        aug_groups_np = np.array(augmented_embeddings_dict["groups"])  # ✅ Directly use stored groups

        # print("!!!AUGMENTED EMBEDDINGS DICT GROUPS (taken from argument): ", Counter(aug_groups_np))

        # print("!!!! aug embeddings np: ", len(aug_embeddings_np))

        if remove_species != None:
          indices = [aug_groups_np[i] not in remove_species for i in range(len(aug_groups_np))]

          #this is to remove species (e.g hyssopifolia) from augmentation
          aug_embeddings_np_reduced = np.array([aug_embeddings_np[i] for i in range(len(indices)) if indices[i]])
          aug_labels_np_reduced = np.array([aug_labels_np[i] for i in range(len(indices)) if indices[i]])
          aug_obs_id_np_reduced = np.array([aug_obs_id_np[i] for i in range(len(indices)) if indices[i]])
          aug_groups_np_reduced = np.array([aug_groups_np[i] for i in range(len(indices)) if indices[i]])

          # print("!!!! Aug embeddings np after removing species !!!! : ", len(aug_embeddings_np))

        else:
          aug_embeddings_np_reduced = aug_embeddings_np
          aug_labels_np_reduced = aug_labels_np
          aug_obs_id_np_reduced = aug_obs_id_np
          aug_groups_np_reduced = aug_groups_np

          # print("!!!! Aug embedings np NOT REMOVING SPECIES: ", len(aug_embeddings_np))

        # Get val groups from main split
        val_groups_set = set(groups_np_reduced[i] for i in val_idx)

        # print("!!! Val groups set: ", val_groups_set)

        # print("!!!Groups np reduced BEFORE MASKING: ", Counter(aug_groups_np_reduced))
        # Keep only augmented samples whose group is NOT in val groups
        aug_mask = np.array([g not in val_groups_set for g in aug_groups_np_reduced])

        # print("!!! Counter of aug_mask: ", Counter(aug_mask))
        # print("Aug mask: ", aug_mask[:10])
        # print("!!! Counter of aug_groups_np_reduced: ", Counter(aug_groups_np_reduced))

        aug_embeddings_np_reduced = aug_embeddings_np_reduced[aug_mask]
        aug_labels_np_reduced = aug_labels_np_reduced[aug_mask]
        aug_obs_id_np_reduced = aug_obs_id_np_reduced[aug_mask]
        aug_groups_np_reduced = aug_groups_np_reduced[aug_mask]

        print("!!!AUG Groups np reduced AFTER MASKING: ", Counter(aug_groups_np_reduced))

        # print("!!!! aug embeddings np AFTER MASKING: ", len(aug_embeddings_np_reduced))

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # or get from caller
        aug_embeddings_tensors = torch.from_numpy(np.stack(aug_embeddings_np_reduced)).to(device)
        aug_labels_tensors = torch.from_numpy(np.array(aug_labels_np_reduced)).to(device)
        aug_obs_id_tensor = torch.from_numpy(np.array(aug_obs_id_np_reduced)).to(device)

        augmented_train_dataset = torch.utils.data.TensorDataset(aug_embeddings_tensors, aug_labels_tensors, aug_obs_id_tensor)
        # print("train dataset before augmentation: ", len(train_dataset))
        # print("!!!!!!!! Only Augmented train dataset length: ", len(augmented_train_dataset))
        train_dataset = ConcatDataset([train_dataset, augmented_train_dataset])

    #class weights (as before)
    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)
    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts.get(0, 1)),
        1.0 / math.log(1.02 + label_counts.get(1, 1)),
    ], dtype=torch.float)
    print("⚖️ Class weights:", class_weights)

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, class_weights, embedding_size


In [16]:
#@title Classifier
class InvasiveSpeciesClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=256, num_classes=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(), #consider adding a Dropout layer
            # nn.Dropout(0.3), #We remove the dropout layer
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [17]:
#@title Early Stopping class

#Patience: how many epochs to wait after last improvements
#delta: minimum change in validation to be considered improvement
class EarlyStopping:

    def __init__(self, patience=10, delta=0.01, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False

    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [18]:
#@title 4. Training function
def train_classifier(train_loader, val_loader, feature_extractor, classifier, optimizer, criterion, device, output_dir, model_name, num_epoch=10, verbose=True, save_classifier=False, do_early_stopping=False):

    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    early_stopping = EarlyStopping(patience=25, delta=0, verbose=True)

    for epoch in range(num_epoch):
        print(f"\n🔁 Epoch {epoch + 1}/{num_epoch}")
        classifier.train()
        running_loss, correct, total = 0, 0, 0

        for embeddings, labels, _ in tqdm(train_loader, desc="Training"): # Unpack all three values
            outputs = classifier(embeddings)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * embeddings.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        val_loss, val_acc, y_true, y_pred = evaluate_classifier(
            val_loader, feature_extractor, classifier, criterion, device
        )

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        if verbose:
            print(f"📉 Train Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
            print(f"📈 Val Loss:   {val_loss:.4f} | Accuracy: {val_acc:.4f}")

            print("\n📋 Classification Report:")
            report_dict = classification_report(y_true, y_pred, labels=[0, 1], target_names=["Invasive", "Non Invasive"], zero_division=0, output_dict=True)
            print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Invasive", "Non Invasive"], zero_division=0, output_dict=False))

            #plot confusion matrix
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            fig, ax = plt.subplots(figsize=(6, 6))
            im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            ax.figure.colorbar(im, ax=ax)
            classes = ["Invasive", "Non Invasive"]

            # Show all ticks and label them
            ax.set(xticks=np.arange(len(classes)),
                yticks=np.arange(len(classes)),
                xticklabels=classes,
                yticklabels=classes,
                ylabel='True label',
                xlabel='Predicted label',
                title='Validation Confusion Matrix')

            # Rotate the tick labels and set alignment.
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

            # Loop over data dimensions and create text annotations.
            fmt = 'd'
            thresh = cm.max() / 2. if cm.max() > 0 else 1
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(j, i, format(cm[i, j], fmt),
                            ha="center", va="center",
                            color="white" if cm[i, j] > thresh else "black")

            fig.tight_layout()
            plt.grid(False)
            plt.show()
        else:
          report_dict = None

        if do_early_stopping:
          # Check early stopping condition
          early_stopping.check_early_stop(val_loss)

          if early_stopping.stop_training:
            print(f"Early stopping at epoch {epoch}")
            break


    torch.save(classifier.state_dict(), f'classifier_head_{model_name}.pth')
    google_drive_path = f'/content/drive/MyDrive/Thesis/species_models/classifier_head_{model_name}.pth'
    torch.save(classifier.state_dict(), google_drive_path)

    return train_losses, train_accuracies, val_losses, val_accuracies, report_dict

In [19]:
#@title 5. Evaluation function

from sklearn.metrics import balanced_accuracy_score

def evaluate_classifier(loader, feature_extractor, classifier, criterion, device, save_csv_path, embeddings_dict):
    classifier.eval()
    total_loss, correct, total = 0, 0, 0


    all_preds = []
    all_labels = []
    all_obs_ids = []
    all_filenames = []
    all_groups = []


    # ✅ NEW: store raw logits and softmax probabilities
    all_logits = []
    all_probs = []
    softmax = torch.nn.Softmax(dim=1)


    with torch.no_grad():
        for embeddings, labels, obs_ids in tqdm(loader, desc="Evaluating"): # Unpack all three values
            embeddings, labels = embeddings.to(device), labels.to(device)
            outputs = classifier(embeddings)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * embeddings.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_obs_ids.extend(obs_ids.cpu().numpy() if torch.is_tensor(obs_ids) else obs_ids)

            # ✅ fetch filenames and groups directly from embeddings_dict
            if embeddings_dict is not None:
                fnames = [embeddings_dict["filenames"][int(oid)] for oid in obs_ids]
                taxa   = [embeddings_dict["groups"][int(oid)] for oid in obs_ids]
                all_filenames.extend(fnames)
                all_groups.extend(taxa)
            else:
                all_filenames.extend([None] * len(obs_ids))
                all_groups.extend([None] * len(obs_ids))

            # ✅ NEW: capture logits and probabilities for this batch
            logits_b = outputs.detach().cpu()
            probs_b  = softmax(outputs).detach().cpu()

            all_logits.extend(logits_b.tolist())
            all_probs.extend(probs_b.tolist())

    avg_loss = total_loss / total
    avg_acc = correct / total

    if save_csv_path:
        # ✅ wide-format CSV: separate columns for each class’s logits/probs
        n_classes = len(all_probs[0]) if len(all_probs) else 0
        base = {
            "obs_id": all_obs_ids,
            "filename": all_filenames,
            "group": all_groups,
            "y_true": all_labels,
            "y_pred": all_preds,
        }
        for i in range(n_classes):
            base[f"logit_c{i}"] = [row[i] for row in all_logits]
            base[f"prob_c{i}"]  = [row[i] for row in all_probs]

        df = pd.DataFrame(base)
        df.to_csv(save_csv_path, index=False, mode='a')  # append instead of overwriting
        print(f"📂 Saved evaluation results to {save_csv_path}")

    return avg_loss, avg_acc, all_labels, all_preds, all_obs_ids, all_filenames

In [20]:
#@title Focal loss function

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        alpha: Class weighting (tensor or list), same as in CrossEntropyLoss
        gamma: Focusing parameter (higher → more focus on hard samples) (gamma=0 => cross entropy loss)
        reduction: 'mean', 'sum', or 'none'
        """
        super(FocalLoss, self).__init__()
        self.alpha=alpha
        self.gamma=gamma
        self.reduction=reduction

    def forward(self, inputs, targets):
        #cross entropy per sample (no reduction)
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')

        #convert to probabilities of true class for modulating factor
        pt = torch.exp(-ce_loss) #ce = -log(pt), so pt = e^(-ce), model confidence of true class

        #focal loss formula (focal term)
        focal_loss = ((1-pt)**self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean() #return average loss per batch
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


In [21]:
#@title Full Model
class FullModel(nn.Module):
    def __init__(self, embedding_model, classifier_head):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier_head = classifier_head

    def forward(self, x):
        with torch.no_grad():
            if hasattr(self.encoder, 'encode_image'):
                features = self.encoder.encode_image(x)
            else:
                features = self.encoder(x)

        if features.ndim == 4:
            features = features.view(features.size(0), -1)

        return self.classifier_head(features)

In [22]:
#@title Load model and predict
def load_model_and_predict(folder_name="taxas", use_bioclip=True, bioclip_version=None,
                             augmentation=True, augmentation_factor=2, loss_type='CE', output_dir="invasive_species",
                             model_name="prova", verbose=True, save_model=True, embeddings_save_path=None,
                             epochs=50, debug_subset_size=None, n_splits=5, fold_idx=0, remove_species=None,
                             do_early_stopping=False, classifier_path=None, predictions_path=None):
    start_time = time.time()

    np.random.seed(42)
    torch.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    github = '/content/ai4biological-pattern/barbara_new'
    github_repository = f'{github}/{folder_name}'

    if not os.path.isdir(github_repository): #check if data has already been downloaded
        !git sparse-checkout set barbara_new/{folder_name}
        !git checkout clean-barbara

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    if use_bioclip:
        feature_extractor, preprocess = get_bioclip_embeddings_extractor_model(device, bioclip_version)
        feature_extractor = feature_extractor.to(device)
    else:
        feature_extractor = get_resnet_embeddings_extractor_model()
        feature_extractor = feature_extractor.to(device)
        preprocess = None
    print("Feature extractor obtained..")

    def count_unique_taxa(dataset):
      groups = [taxon for taxon, _ in dataset.entries]
      unique_groups = set(groups)
      print(f"📊 Total number of taxa (groups): {len(unique_groups)}")
      return unique_groups

    #Load dataset to reconstruct groups
    dataset = InvasiveSpeciesDataset(root_dir=github_repository, transform=preprocess if use_bioclip else None)
    # inspect_folds_groups(dataset, n_splits=n_splits)
    unique_taxa = count_unique_taxa(dataset)

    if os.path.exists(embeddings_save_path):

        print("✅ Embeddings already exist, loading from file...")
        embeddings_dict = torch.load(embeddings_save_path)

        img_loader = get_data_loader(root_dir=github_repository)

        if augmentation:
          #only extract embeddings for augmentated data
          augmented_loader = get_data_loader_augmented(root_dir=github_repository, augmentation='flipping')
          augmented_embeddings_dict = extract_embeddings(augmented_loader, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

          if augmentation_factor==3:
            augmented_loader_2 = get_data_loader_augmented(root_dir=github_repository, augmentation='jittering')
            augmented_embeddings_dict_2 = extract_embeddings(augmented_loader_2, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

            augmented_embeddings_dict = {
              "embeddings": torch.cat([augmented_embeddings_dict["embeddings"], augmented_embeddings_dict_2["embeddings"]]),
              "labels": torch.cat([augmented_embeddings_dict["labels"], augmented_embeddings_dict_2["labels"]]),
              "ids": torch.cat([augmented_embeddings_dict["ids"], augmented_embeddings_dict_2["ids"]]),
              "groups": augmented_embeddings_dict["groups"] + augmented_embeddings_dict_2["groups"]
            }

        else:
          augmented_embeddings_dict=None

        if fold_idx==0: #only need it one time
          inspect_folds_groups_with_augmentation(embeddings_dict, augmented_embeddings_dict, dataset, n_splits=n_splits, remove_species=remove_species)

        # Debug the specific fold you're training on
        taxa = debug_validation_set_actual_with_augmentation(embeddings_dict, augmented_embeddings_dict, dataset,
                                                    n_splits=n_splits, fold_idx=fold_idx, remove_species=remove_species)

        model_name = f'bioclip_2_{taxa}' if use_bioclip and bioclip_version==2 else f'bioclip_1_{taxa}' if use_bioclip else f'resnet_{taxa}'


        if debug_subset_size is not None:
          print(f"⚠ Debug mode: using only the first {debug_subset_size} embeddings.")
          embeddings_dict['embeddings'] = embeddings_dict['embeddings'][:debug_subset_size]
          embeddings_dict['labels'] = embeddings_dict['labels'][:debug_subset_size]
          embeddings_dict['ids'] = embeddings_dict['ids'][:debug_subset_size]

          if augmentation:
            augmented_embeddings_dict['embeddings'] = augmented_embeddings_dict['embeddings'][:debug_subset_size]
            augmented_embeddings_dict['labels'] = augmented_embeddings_dict['labels'][:debug_subset_size]
            augmented_embeddings_dict['ids'] = augmented_embeddings_dict['ids'][:debug_subset_size]
          else:
            augmented_embeddings_dict=None


        embeddings_train_loader, embeddings_val_loader, class_weights, input_size = split_embeddings_groupkfold(
               embeddings_dict, augmented_embeddings_dict, n_splits=n_splits, fold_idx=fold_idx, remove_species=remove_species
        )

    else:

        img_loader = get_data_loader(root_dir=github_repository)
        print("Dataloader obtained..")

        #this function also saves embeddings in the desired path
        #input size is the same for training and validation, is is the size (number of features) of the single embedding.
        embeddings_dict = extract_embeddings(img_loader, feature_extractor, device, use_bioclip=use_bioclip, embeddings_save_path=embeddings_save_path, dataset=dataset)

        if augmentation:
          #only extract embeddings for augmentated data
          augmented_loader = get_data_loader_augmented(root_dir=github_repository)
          augmented_embeddings_dict = extract_embeddings(augmented_loader, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

          if augmentation_factor==3:
            augmented_loader_2 = get_data_loader_augmented(root_dir=github_repository, augmentation='jittering')
            augmented_embeddings_dict_2 = extract_embeddings(augmented_loader_2, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

            augmented_embeddings_dict = {
              "embeddings": torch.cat([augmented_embeddings_dict["embeddings"], augmented_embeddings_dict_2["embeddings"]]),
              "labels": torch.cat([augmented_embeddings_dict["labels"], augmented_embeddings_dict_2["labels"]]),
              "ids": torch.cat([augmented_embeddings_dict["ids"], augmented_embeddings_dict_2["ids"]]),
              "groups": augmented_embeddings_dict["groups"] + augmented_embeddings_dict_2["groups"]
            }

        else:
          augmented_embeddings_dict=None

        if debug_subset_size is not None:
          print(f"⚠ Debug mode: using only the first {debug_subset_size} embeddings.")
          embeddings_dict['embeddings'] = embeddings_dict['embeddings'][:debug_subset_size]
          embeddings_dict['labels'] = embeddings_dict['labels'][:debug_subset_size]
          embeddings_dict['ids'] = embeddings_dict['ids'][:debug_subset_size]

          if augmentation:
            augmented_embeddings_dict['embeddings'] = augmented_embeddings_dict['embeddings'][:debug_subset_size]
            augmented_embeddings_dict['labels'] = augmented_embeddings_dict['labels'][:debug_subset_size]
            augmented_embeddings_dict['ids'] = augmented_embeddings_dict['ids'][:debug_subset_size]
          else:
            augmented_embeddings_dict=None

        embeddings_train_loader, embeddings_val_loader, class_weights, input_size = split_embeddings_groupkfold(
               embeddings_dict, augmented_embeddings_dict, n_splits=n_splits, fold_idx=fold_idx
        )


    classifier = InvasiveSpeciesClassifier(embedding_dim=input_size).to(device)

    #after initialization, load finetuned classifier
    print("Loading classifier...")
    classifier.load_state_dict(torch.load(f'/content/drive/MyDrive/Thesis/species_models/classifier_head_{model_name}.pth'))
    print("Classifier loaded")

    if loss_type=='CE':
      criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    elif loss_type=='FL':
      criterion = FocalLoss(alpha=class_weights.to(device), gamma=2, reduction='mean')
    optimizer = optim.Adam(classifier.parameters(), lr=1e-4)

    avg_loss, avg_acc, all_labels, all_preds, all_obs_ids, all_filenames = evaluate_classifier(embeddings_val_loader, feature_extractor, classifier, criterion, device, predictions_path, embeddings_dict)

    # full_model = FullModel(feature_extractor, classifier)
    # print("Full model built...")
    # full_model.eval()

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Total execution time: {elapsed_time:.2f} seconds")

    return taxa

# Model executions

## **Testing without Hyssopifolia** (with more determinism than before)
#### See if results change much

In [23]:
remove_species=['lythrum_hyssopifolia']

n_splits = 29

# embeddings_save_path_BC2_new = "/content/drive/MyDrive/Thesis/bioclip2_groups_embeddings_v0.pt"
embeddings_save_path_BC2 = '/content/drive/MyDrive/Thesis/bioclip2_groups_embeddings_with_filenames.pt'
model_name_final = 'group_k_fold_new_embeddings'

predictions_path = '/content/drive/MyDrive/Thesis/full_predictions_with_logits.csv'

epochs=200

for fold_idx in range(n_splits):
  taxa = load_model_and_predict(folder_name="taxas", use_bioclip=True, augmentation=False, augmentation_factor=2,
                                          loss_type='CE', bioclip_version=2, output_dir="invasive_species", model_name=model_name_final,
                                           save_model=True, verbose=False, embeddings_save_path=embeddings_save_path_BC2,
                                             epochs=epochs, debug_subset_size=None, n_splits=n_splits, fold_idx=fold_idx,
                                              remove_species=remove_species, do_early_stopping=True, predictions_path=predictions_path)

  print(f"✅ Fold {fold_idx} done ({taxa})")

Updating files: 100% (45475/45475), done.
Already on 'clean-barbara'
Your branch is up to date with 'origin/clean-barbara'.
Using device: cuda
BioCLIP 2!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Model obtained...
Feature extractor obtained..
📊 Total number of taxa (groups): 30
✅ Embeddings already exist, loading from file...
🔍 Inspecting 29-fold GroupKFold with augmentation effects:
📊 Total samples in filtered embeddings: 35778
📊 Total unique groups: 29

📂 Fold 0: 1 validation groups
VALIDATION - Class distribution: {np.int64(0): 13105}
TRAINING (original) - Class distribution: {np.int64(1): 20528, np.int64(0): 2145}
Species in validation fold: [np.str_('lythrum_salicaria')]

📂 Fold 1: 1 validation groups
VALIDATION - Class distribution: {np.int64(1): 10491}
TRAINING (original) - Class distribution: {np.int64(0): 15250, np.int64(1): 10037}
Species in validation fold: [np.str_('lythrum_alatum')]

📂 Fold 2: 1 validation groups
VALIDATION - Class distribution: {np.int64(1): 3290}
TRAINING (original) - Class distribution: {np.int64(0): 15250, np.int64(1): 17238}
Species in validation fold: [np.str_('lythrum_junceum')]

📂 Fold 3: 1 validation groups
VALIDATION - Class distribution:

...

Evaluating: 100%|██████████| 1/1 [00:00<00:00, 461.12it/s]

📂 Saved evaluation results to /content/drive/MyDrive/Thesis/full_predictions_with_logits.csv
Total execution time: 11.32 seconds
✅ Fold 27 done (lythrum_silenoides)
Using device: cuda
BioCLIP 2!


Model obtained...
Feature extractor obtained..
📊 Total number of taxa (groups): 30
✅ Embeddings already exist, loading from file...

🔍 DEBUG: Actual sets for fold 28 (lythrum_thesioides):
VALIDATION SET:
  Size: 1
  Class distribution: {np.int64(1): 1}
  Unique groups: [np.str_('lythrum_thesioides')]
TRAINING SET (original):
  Size: 35777
  Class distribution: {np.int64(0): 15250, np.int64(1): 20527}
  Unique groups: [np.str_('lythrum_acutangulum'), np.str_('lythrum_alatum'), np.str_('lythrum_album'), np.str_('lythrum_baeticum'), np.str_('lythrum_borysthenicum'), np.str_('lythrum_bryantii'), np.str_('lythrum_californicum'), np.str_('lythrum_curtissii'), np.str_('lythrum_flagellare'), np.str_('lythrum_flexuosum'), np.str_('lythrum_gracile'), np.str_('lythrum_intermedium'), np.str_('lythrum_junceum'), np.str_('lythrum_lineare'), np.str_('lythrum_maritimum'), np.str_('lythrum_netofa'), np.str_('lythrum_ovalifolium'), np.str_('lythrum_paradoxum'), np.str_('lythrum_portula'), np.str_('lythr

Evaluating: 100%|██████████| 1/1 [00:00<00:00, 650.08it/s]

📂 Saved evaluation results to /content/drive/MyDrive/Thesis/full_predictions_with_logits.csv
Total execution time: 10.77 seconds
✅ Fold 28 done (lythrum_thesioides)


In [24]:
#@title prediction only for hyssopifolia (fold id = 2)
n_splits = 30

# embeddings_save_path_BC2_new = "/content/drive/MyDrive/Thesis/bioclip2_groups_embeddings_v0.pt"
embeddings_save_path_BC2 = '/content/drive/MyDrive/Thesis/bioclip2_groups_embeddings_with_filenames.pt'
model_name_final = 'group_k_fold_new_embeddings'

predictions_path = '/content/drive/MyDrive/Thesis/hyssopifolia_predictions_with_logits.csv'

epochs=200


taxa = load_model_and_predict(folder_name="taxas", use_bioclip=True, augmentation=False, augmentation_factor=2,
                                          loss_type='CE', bioclip_version=2, output_dir="invasive_species", model_name=model_name_final,
                                           save_model=True, verbose=False, embeddings_save_path=embeddings_save_path_BC2,
                                             epochs=epochs, debug_subset_size=None, n_splits=n_splits, fold_idx=2,
                                              remove_species=None, do_early_stopping=True, predictions_path=predictions_path)

print(f"✅ Fold {2} done ({taxa})")

Using device: cuda
BioCLIP 2!
Model obtained...
Feature extractor obtained..
📊 Total number of taxa (groups): 30
✅ Embeddings already exist, loading from file...

🔍 DEBUG: Actual sets for fold 2 (lythrum_hyssopifolia):
VALIDATION SET:
  Size: 9583
  Class distribution: {np.int64(0): 9583}
  Unique groups: [np.str_('lythrum_hyssopifolia')]
TRAINING SET (original):
  Size: 35778
  Class distribution: {np.int64(0): 15250, np.int64(1): 20528}
  Unique groups: [np.str_('lythrum_acutangulum'), np.str_('lythrum_alatum'), np.str_('lythrum_album'), np.str_('lythrum_baeticum'), np.str_('lythrum_borysthenicum'), np.str_('lythrum_bryantii'), np.str_('lythrum_californicum'), np.str_('lythrum_curtissii'), np.str_('lythrum_flagellare'), np.str_('lythrum_flexuosum'), np.str_('lythrum_gracile'), np.str_('lythrum_intermedium'), np.str_('lythrum_junceum'), np.str_('lythrum_lineare'), np.str_('lythrum_maritimum'), np.str_('lythrum_netofa'), np.str_('lythrum_ovalifolium'), np.str_('lythrum_paradoxum'), np.

Evaluating: 100%|██████████| 300/300 [00:00<00:00, 563.76it/s]


📂 Saved evaluation results to /content/drive/MyDrive/Thesis/hyssopifolia_predictions_with_logits.csv
Total execution time: 12.47 seconds
✅ Fold 2 done (lythrum_hyssopifolia)
